In [76]:
import json
import yaml
from mstrio.connection import Connection
from mstr_robotics.mstr_classes import mstr_global
from mstr_robotics.redis_db import  redis_bi_analysis,redis_mstr_json
from mstr_robotics._connectors import mstr_api
from mstr_robotics._helper import msic
from mstr_robotics.read_out_prj_obj import read_gen
from mstr_robotics.prepare_AI_data import export_mstr_md

i_mstr_global=mstr_global()
i_mstr_api=mstr_api()
i_redis_mstr_json=redis_mstr_json()
i_export_mstr_md=export_mstr_md()
i_read_gen=read_gen()
i_msic=msic()

with open('..\\config\\mstr_redis_y.yml', 'r') as openfile:
    mstr_redis_y = yaml.safe_load(openfile)

In [77]:
redis_con_d=mstr_redis_y["redis_env_d"]["redis_dev"]
project_prefix=mstr_redis_y["project_prefix"]
prefix_map=mstr_redis_y["prefix_map"]
searches_used_in_prp_d_l=mstr_redis_y["searches_used_in_prp_d_l"]


## Connect to MSTR & Redis

In [78]:
with open('..\\config\\user_d.json', 'r') as openfile:
    user_d = json.load(openfile)
conn_params =  user_d["conn_params"]
conn = Connection(**conn_params)
conn.headers['Content-type'] = "application/json"

i_redis_bi_analysis = redis_bi_analysis( 
    host=redis_con_d["host"],
    port=redis_con_d["port"],
    password=redis_con_d["password"],
    username=redis_con_d["username"],
    decode_responses=redis_con_d["decode_responses"]
)


Connection to Strategy One Intelligence Server has been established.
No project selected.


### Daily Search Update

In [ ]:

err_d_l=[]
for pre in project_prefix:
    env_prefix=project_prefix[pre]
    conn.select_project(pre)
    search_result = i_mstr_api.run_mstr_search(conn=conn,
                        search_id="96648F2B492150A6AA27DDB3744E32B4")
    print(search_result)

    if search_result["totalItems"] > 0:
        all_obj_d_l=search_result["result"]
        """
        err_d_l.append(i_redis_mstr_json.save_obj_json_to_redis(i_redis_bi_analysis=i_redis_bi_analysis                                                    
                                                        ,conn=conn
                                                        ,prefix_map=prefix_map
                                                        , all_obj_d_l=all_obj_d_l
                                                        , env_prefix=env_prefix)
                        )
        print(str(all_obj_d_l) + " uploaded to " + env_prefix)
        """
all_obj_d_l

## Fetch MSTR objects

### report tables

In [79]:
redis_env_p="mstr_dev"
redis_pre_obj="TABLE"
table_key_l=[]
table_d_l=[{"id":"24C30AD611D5AEC9C000E38A4CC5F24F","name":"lu_day"},
          {"id":"8D67933211D3E4981000E787EC6DE8A4","name":"lu_call_ctr"},
          {"id":"8D67933E11D3E4981000E787EC6DE8A4","name":"lu_category"},
          {"id":"8D67936811D3E4981000E787EC6DE8A4","name":"lu_employee"},
          {"id":"8D67937411D3E4981000E787EC6DE8A4","name":"lu_item"},
          {"id":"8D67938011D3E4981000E787EC6DE8A4","name":"lu_month"},
          {"id":"8D6793AA11D3E4981000E787EC6DE8A4","name":"lu_region"},
          {"id":"8D6793B611D3E4981000E787EC6DE8A4","name":"lu_subcateg"},
          {"id":"8D6793CE11D3E4981000E787EC6DE8A4","name":"order_detail"}]
for table in table_d_l:
    oby_key=f'{redis_env_p}:{redis_pre_obj}:{table["id"]}'
    table_key_l.append(oby_key)

### Fetch Dependencies for Specific Report

In [80]:
from mstr_robotics.redis_db import fetch_it_all

# Initialize the fetch_it_all class with your Redis connection
i_fetch_it_all = fetch_it_all(i_redis_bi_analysis)

# Define the root object key for the report definition you want to fetch
root_object_key = "mstr_dev:REPORT_DEFINITION:66FFEB6E49634F501160D2A4FB9BD78A"

# Fetch all dependencies recursively
all_dependencies = i_fetch_it_all.fetch_all_objects_recursively(
    root_object_l=[root_object_key],
    recursive_fg=True,
    batch_size=100
)



In [ ]:
direct_obj_check_d_l=[]
obj_type_set=set()
obj_to_export_d_l=[]
for obj in all_dependencies:
    try:
        if "information" in obj["definition"].keys():
            obj_type_set.add(obj["definition"]["information"]["subType"])
            if obj["definition"]["information"]["subType"] in ["report_grid","agg_metric"]:
                direct_obj_check_l=i_fetch_it_all.fetch_all_objects_recursively(root_object_l=[obj["obj_key"]]
                                             , recursive_fg=False)
                direct_obj_check_d_l.extend(direct_obj_check_l)
            if obj["definition"]["information"]["subType"] in ["report_grid","agg_metric"]:
                obj_to_export_d_l.extend([obj["obj_key"]])

        else:
            obj_type_set.add(obj["definition"]["subType"])
            if obj["definition"]["subType"] in ["metric","agg_metric"]:
                direct_obj_check_l=i_fetch_it_all.fetch_all_objects_recursively(root_object_l=[obj["obj_key"]]
                                             , recursive_fg=False)
                direct_obj_check_d_l.extend(direct_obj_check_l)
            if obj["definition"]["subType"] in ["metric","filter","prompt"]:
                obj_to_export_d_l.extend([obj["obj_key"]])
                
    except:
        print("rreee")
        print(obj["obj_key"])

schema_obj_l=[]
for obj in direct_obj_check_d_l:
    try:
        if "information" in obj["definition"].keys():
            if obj["definition"]["information"]["subType"] in ["attribute","agg_metric"]:
                schema_obj_l.append(obj["obj_key"])
        
        else:
            #print(obj["definition"]["subType"])
            if obj["definition"]["subType"] in ["attribute","fact","hierarchy"]:
                schema_obj_l.append(obj["obj_key"])
    except:
        print("rreee")

schema_obj_l=i_msic.rem_dbl_in_l(schema_obj_l)
#schema_obj_d_l = remove_duplicates_by_obj_id(schema_obj_d_l)
obj_to_export_d_l.extend(schema_obj_l)
obj_to_export_d_l.extend(table_key_l)
obj_to_export_d_l


In [83]:
conn.select_project("B7CA92F04B9FAE8D941C3E9B7E0CD754")
obj_id_l=[]
obj_def_d_l=[]
for obj in obj_to_export_d_l:
    obj_id_l.append(obj.split(":")[2])
obj_id_d_l=i_mstr_api.get_proj_obj_by_id_l(conn=conn, obj_id_l=obj_id_l)
for o in obj_id_d_l:
    obj_def_d_l.append(i_read_gen.get_obj_def(conn=conn,object_id=o["id"],
                                              obj_type=o["obj_type"],
                                              obj_sub_type=o["subtype"]))
obj_def_d_l


[{'information': {'dateCreated': '2026-01-08T09:29:24.168Z',
   'dateModified': '2026-01-08T10:55:19.355Z',
   'versionId': '7D7304514291A0708B0E749132A786B5',
   'acg': 255,
   'primaryLocale': 'en-US',
   'objectId': '66FFEB6E49634F501160D2A4FB9BD78A',
   'subType': 'report_grid',
   'name': 'SchemaTransferReport'},
  'sourceType': 'normal',
  'dataSource': {'dataTemplate': {'units': [{'id': '8D679D5111D3E4981000E787EC6DE8A4',
      'name': 'Year',
      'type': 'attribute',
      'nonAggregatable': False},
     {'id': '8D679D4411D3E4981000E787EC6DE8A4',
      'name': 'Month',
      'type': 'attribute',
      'nonAggregatable': False},
     {'id': '96ED3EC811D5B117C000E78A4CC5F24F',
      'name': 'Day',
      'type': 'attribute',
      'nonAggregatable': False},
     {'id': '8D679D4B11D3E4981000E787EC6DE8A4',
      'name': 'Region',
      'type': 'attribute',
      'nonAggregatable': False},
     {'id': '8D679D3511D3E4981000E787EC6DE8A4',
      'name': 'Call Center',
      'type': 'a

In [85]:
table_d_l=[]
for ob in obj_def_d_l:
   if "subType" in ob.keys():
      if ob["subType"] == "logical_table":
        table_d_l.append(ob)
table_d_l[0]  
  

{'physicalTable': {'information': {'dateCreated': '2001-01-02T20:48:53.000Z',
   'dateModified': '2021-12-09T19:24:15.285Z',
   'versionId': 'E9AA8F7091443D0A5E7A8A83541B7299',
   'acg': 255,
   'primaryLocale': 'en-US',
   'objectId': '8D67910111D3E4981000E787EC6DE8A4',
   'subType': 'physical_table',
   'name': 'lu_call_ctr'},
  'tableName': 'lu_call_ctr',
  'columns': [{'information': {'dateCreated': '2001-01-02T20:48:32.000Z',
     'dateModified': '2025-11-27T11:41:20.638Z',
     'versionId': 'F6719F6D1D4314721C40B6B4D5AFF348',
     'acg': 255,
     'primaryLocale': 'en-US',
     'objectId': '8D67917E11D3E4981000E787EC6DE8A4',
     'subType': 'column',
     'name': 'call_ctr_id'},
    'dataType': {'type': 'integer', 'precision': 2, 'scale': -2147483648},
    'columnName': 'call_ctr_id'},
   {'information': {'dateCreated': '2001-01-02T20:48:31.000Z',
     'dateModified': '2025-11-27T11:41:20.638Z',
     'versionId': 'DF698BA582427CFF5AAABAAD18B86450',
     'acg': 255,
     'primaryL

In [ ]:
dim_table=""
from mstr_robotics.user_RAG import perplexity
from dotenv import load_dotenv
env_file="..\\config\\streamlit.env"
load_dotenv(env_file)

u_perplexity=perplexity()
msg_t=str( table_d_l[:2])

sys_cont=f"Please generate me valid dataset files, which support the AtScale system."
#sys_cont+=f" you find the valid definition und {def_tables} "
sys_cont+=f" for detailed instructions, please check this gitub page https://github.com/semanticdatalayer/SML/blob/main/sml-reference/dataset.md"
sys_cont+=f" in the user message, you find logical table definitions from MicroStrategy"
sys_cont+=" Please do not comment at all. what I need is a clear and well-structured dataset file."


message_check_d={}

dim_table=u_perplexity.call_perplexity( msg_t=msg_t, sys_cont=sys_cont, message_check_d=message_check_d, temperature=0.1)


invalid syntax (<unknown>, line 1)


In [86]:
import os

from openai import OpenAI
temperature=1
client = OpenAI(
    api_key=os.environ.get("PERPLEXITY_API_KEY"),
    # Set your API key in environment variables
    base_url="https://api.perplexity.ai"
)
try:
    # Create chat completion request
    messages = [
        {
            "role": "system",
            "content": sys_cont
        },
        {"role": "user", "content": msg_t}
    ]
    # print(messages)
    response = client.chat.completions.create(
        model="sonar-pro",  # Official model name for Perplexity-API
        messages=messages,
        temperature=temperature
    )
except Exception as e:
    print(f"An error occurred: {e}")

In [87]:
#jj=json.loads(response.json())["choices"][0]["message"]["content"]
import re
import yaml
yaml_string=jj
clean_yaml = re.sub(r'^```yaml\s*\n', '', yaml_string)
clean_yaml = re.sub(r'\n```$', '', clean_yaml)
#documents_list = list(yaml.safe_load_all(clean_yaml))
#documents_list
# Write to a specific directory
with open(r'C:\coding\python_io\output_files\SML\clean_yaml.md', 'w', encoding='utf-8') as f:
    f.write(clean_yaml)

In [41]:
def_tables="""unique_name: store_sales
object_type: dataset
label: store_sales
connection_id: Connection - TPCDS
table: store_sales

columns:
  - name: Net Profit Tier
    data_type: string
    sql: 'CASE WHEN "ss_net_profit" > 25000 THEN ''More than 25000''
      WHEN "ss_net_profit" BETWEEN 3000 AND 25000 THEN ''3000-25000''
      WHEN "ss_net_profit" BETWEEN 2000 AND 3000 THEN ''2000-3000''
      WHEN "ss_net_profit" BETWEEN 300 AND 2000 THEN ''300-2000''
      WHEN "ss_net_profit" BETWEEN 250 AND 300 THEN ''250-300''
      WHEN "ss_net_profit" BETWEEN 200 AND 250 THEN ''200-250''
      WHEN "ss_net_profit" BETWEEN 150 AND 200 THEN ''150-200''
      WHEN "ss_net_profit" BETWEEN 100 AND 150 THEN ''100-150''
      WHEN "ss_net_profit" BETWEEN 50 AND 100 THEN '' 50-100''
      WHEN "ss_net_profit" BETWEEN 0 AND 50 THEN ''  0- 50''
      ELSE '' 50 or Less''
      END'
    dialects:
      - dialect: DatabricksSQL
        sql: "CASE WHEN ss_net_profit > 25000 THEN 'More than 25000'
          WHEN ss_net_profit BETWEEN 3000 AND 25000 THEN '3000-25000'
          WHEN ss_net_profit BETWEEN 2000 AND 3000 THEN '2000-3000'
          WHEN ss_net_profit BETWEEN 300 AND 2000 THEN '300-2000'
          WHEN ss_net_profit BETWEEN 250 AND 300 THEN '250-300'
          WHEN ss_net_profit BETWEEN 200 AND 250 THEN '200-250'
          WHEN ss_net_profit BETWEEN 150 AND 200 THEN '150-200'
          WHEN ss_net_profit BETWEEN 100 AND 150 THEN '100-150'
          WHEN ss_net_profit BETWEEN 50 AND 100 THEN ' 50-100'
          WHEN ss_net_profit BETWEEN 0 AND 50 THEN '  0- 50'
          ELSE ' 50 or Less'
          END"
      - dialect: BigQuery
        sql: "CASE WHEN ss_net_profit > 25000 THEN 'More than 25000'
          WHEN ss_net_profit BETWEEN 3000 AND 25000 THEN '3000-25000'
          WHEN ss_net_profit BETWEEN 2000 AND 3000 THEN '2000-3000'
          WHEN ss_net_profit BETWEEN 300 AND 2000 THEN '300-2000'
          WHEN ss_net_profit BETWEEN 250 AND 300 THEN '250-300'
          WHEN ss_net_profit BETWEEN 200 AND 250 THEN '200-250'
          WHEN ss_net_profit BETWEEN 150 AND 200 THEN '150-200'
          WHEN ss_net_profit BETWEEN 100 AND 150 THEN '100-150'
          WHEN ss_net_profit BETWEEN 50 AND 100 THEN ' 50-100'
          WHEN ss_net_profit BETWEEN 0 AND 50 THEN '  0- 50'
          ELSE ' 50 or Less'
          END"
  - name: Purchased Amount in Store
    data_type: "decimal(16,8)"
    sql: '(("ss_ext_list_price"-"ss_ext_wholesale_cost"-"ss_ext_discount_amt")+"ss_ext_sales_price")/2'
    dialects:
      - dialect: DatabricksSQL
        sql: "((ss_ext_list_price-ss_ext_wholesale_cost-ss_ext_discount_amt)+ss_ext_sales_price)/2"
      - dialect: BigQuery
        sql: "((ss_ext_list_price-ss_ext_wholesale_cost-ss_ext_discount_amt)+ss_ext_sales_price)/2"
  - name: ss row counter
    data_type: int
    sql: "1"
  - name: ss_addr_sk
    data_type: long
  - name: ss_cdemo_sk
    data_type: long
  - name: ss_coupon_amt
    data_type: "decimal(7,2)"
  - name: ss_customer_sk
    data_type: long
  - name: ss_ext_discount_amt
    data_type: "decimal(7,2)"
  - name: ss_ext_list_price
    data_type: "decimal(7,2)"
  - name: ss_ext_sales_price
    data_type: "decimal(7,2)"
  - name: ss_ext_tax
    data_type: "decimal(7,2)"
  - name: ss_ext_wholesale_cost
    data_type: "decimal(7,2)"
  - name: ss_hdemo_sk
    data_type: long
  - name: ss_item_sk
    data_type: long
  - name: ss_list_price
    data_type: "decimal(7,2)"
  - name: ss_net_paid
    data_type: "decimal(7,2)"
  - name: ss_net_paid_inc_tax
    data_type: "decimal(7,2)"
  - name: ss_net_profit
    data_type: "decimal(7,2)"
  - name: ss_promo_sk
    data_type: long
  - name: ss_quantity
    data_type: long
  - name: ss_sales_price
    data_type: "decimal(7,2)"
  - name: ss_sold_date_sk
    data_type: long
  - name: ss_sold_time_sk
    data_type: long
  - name: ss_store_sk
    data_type: long
  - name: ss_ticket_number
    data_type: long
  - name: ss_wholesale_cost
    data_type: "decimal(7,2)"
  - name: sales price tier
    data_type: string
    sql: 'CASE WHEN "ss_sales_price" > 200 THEN ''200 and More''
      WHEN "ss_sales_price" BETWEEN 150 AND 200 THEN ''150-200''
      WHEN "ss_sales_price" BETWEEN 100 AND 150 THEN ''100-150''
      WHEN "ss_sales_price" BETWEEN 50 AND 100 THEN '' 50-100''
      ELSE '' 50 and Less'' END'
    dialects:
      - dialect: DatabricksSQL
        sql: "CASE WHEN ss_sales_price > 200 THEN '200 and More'
          WHEN ss_sales_price BETWEEN 150 AND 200 THEN '150-200'
          WHEN ss_sales_price BETWEEN 100 AND 150 THEN '100-150'
          WHEN ss_sales_price BETWEEN 50 AND 100 THEN ' 50-100'
          ELSE ' 50 and Less' END"
      - dialect: BigQuery
        sql: "CASE WHEN ss_sales_price > 200 THEN '200 and More'
          WHEN ss_sales_price BETWEEN 150 AND 200 THEN '150-200'
          WHEN ss_sales_price BETWEEN 100 AND 150 THEN '100-150'
          WHEN ss_sales_price BETWEEN 50 AND 100 THEN ' 50-100'
          ELSE ' 50 and Less' END"
"""

'```json\n{\n  "$schema": "https://github.com/semanticdatalayer/SML/blob/main/sml-reference/dataset.schema.json",\n  "name": "MicroStrategy Tutorial Dataset",\n  "description": "Dataset derived from MicroStrategy Tutorial logical tables for AtScale compatibility",\n  "version": "1.0.0",\n  "tables": [\n    {\n      "name": "LU_CALL_CTR",\n      "description": "Call Center lookup table",\n      "primaryKey": ["call_ctr_id"],\n      "columns": [\n        {\n          "name": "call_ctr_id",\n          "type": "integer",\n          "description": "Call center ID (key form)",\n          "primaryKey": true\n        },\n        {\n          "name": "center_name",\n          "type": "string",\n          "length": 50,\n          "description": "Call center name (DESC form)"\n        },\n        {\n          "name": "country_id",\n          "type": "integer",\n          "description": "Foreign key to Country"\n        },\n        {\n          "name": "region_id",\n          "type": "integer",\n 